In [1]:
import pandas as pd
import numpy as np

sales = pd.read_csv(
    "../data/synthetic/daily_sales.csv",
    parse_dates=["date"]
)

products = pd.read_csv(
    "../data/synthetic/products.csv"
)

print(sales.shape)
print(sales.head())

(73100, 6)
        date product_id medicine_name  quantity_sold  unit_price  revenue
0 2024-01-01     MED001   Paracetamol             20         122     2440
1 2024-01-02     MED001   Paracetamol             42         122     5124
2 2024-01-03     MED001   Paracetamol             26         122     3172
3 2024-01-04     MED001   Paracetamol             36         122     4392
4 2024-01-05     MED001   Paracetamol             28         122     3416


In [2]:
sales["date"] = pd.to_datetime(sales["date"])

daily_sales = (
    sales.groupby(
        ["date", "product_id"],
        as_index=False
    )["quantity_sold"]
    .sum()
)

print(daily_sales.head())

        date product_id  quantity_sold
0 2024-01-01     MED001             20
1 2024-01-01     MED002             13
2 2024-01-01     MED003             28
3 2024-01-01     MED004             26
4 2024-01-01     MED005             15


In [3]:
# Create daily series for every product
start_date = pd.Timestamp("2024-01-01")
end_date = pd.Timestamp("2025-12-31")

all_dates = pd.date_range(
    start=start_date,
    end=end_date,
    freq="D"
)

product_map = products.set_index("product_id")

feature_rows = []

for product_id, group in daily_sales.groupby("product_id"):

    series = (
        group.set_index("date")["quantity_sold"]
        .reindex(all_dates, fill_value=0)
    )

    for cutoff in pd.date_range(
        "2024-01-31",
        "2025-10-31",
        freq="ME"
    ):

        # Historical data available at prediction time
        history = series.loc[:cutoff]

        # Future 30-day target
        future = series.loc[
            cutoff + pd.Timedelta(days=1):
            cutoff + pd.Timedelta(days=30)
        ]

        if len(history) < 90 or len(future) != 30:
            continue

        product_info = product_map.loc[product_id]

        feature_rows.append({

            "forecast_origin": cutoff,
            "product_id": product_id,

            "category": product_info["category"],

            "sales_last_7d": history.tail(7).sum(),

            "sales_last_30d": history.tail(30).sum(),

            "sales_last_90d": history.tail(90).sum(),

            "avg_daily_30d": history.tail(30).mean(),

            "std_daily_30d": history.tail(30).std(),

            "month": (
                cutoff + pd.Timedelta(days=1)
            ).month,

            "target_next_30d": future.sum()
        })

demand_df = pd.DataFrame(feature_rows)

print("Prepared dataset:", demand_df.shape)

demand_df.head()

Prepared dataset: (2000, 10)


,forecast_origin,product_id,category,sales_last_7d,sales_last_30d,sales_last_90d,avg_daily_30d,std_daily_30d,month,target_next_30d
0,2024-03-31,MED001,Analgesic,233,943,2879,31.433333,5.975140,4,925
1,2024-04-30,MED001,Analgesic,197,925,2832,30.833333,5.890280,5,893
2,2024-05-31,MED001,Analgesic,217,901,2771,30.033333,6.110947,6,1206
3,2024-06-30,MED001,Analgesic,264,1206,3028,40.200000,7.720997,7,1133
4,2024-07-31,MED001,Analgesic,241,1130,3250,37.666667,6.293584,8,1221


In [4]:
demand_df["forecast_origin"] = pd.to_datetime(
    demand_df["forecast_origin"]
)

train_df = demand_df[
    demand_df["forecast_origin"] <= "2024-12-31"
]

val_df = demand_df[
    (demand_df["forecast_origin"] >= "2025-01-01") &
    (demand_df["forecast_origin"] <= "2025-06-30")
]

test_df = demand_df[
    (demand_df["forecast_origin"] >= "2025-07-01") &
    (demand_df["forecast_origin"] <= "2025-10-31")
]

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

Training: (1000, 10)
Validation: (600, 10)
Testing: (400, 10)


In [5]:
import os

os.makedirs("../data/processed", exist_ok=True)

train_df.to_csv(
    "../data/processed/train.csv",
    index=False
)

val_df.to_csv(
    "../data/processed/validation.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/test.csv",
    index=False
)

demand_df.to_csv(
    "../data/processed/demand_features.csv",
    index=False
)

print("All datasets saved successfully!")

All datasets saved successfully!


In [6]:
feature_columns = [
    "sales_last_7d",
    "sales_last_30d",
    "sales_last_90d",
    "avg_daily_30d",
    "std_daily_30d",
    "month",
    "category",
    "product_id"
]

target_column = "target_next_30d"

assert target_column not in feature_columns

assert train_df["forecast_origin"].max() < \
       val_df["forecast_origin"].min()

assert val_df["forecast_origin"].max() < \
       test_df["forecast_origin"].min()

print("Leakage checks passed!")

Leakage checks passed!
